<a href="https://colab.research.google.com/github/satani99/triton_kernels/blob/main/intro_to_triton.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
import os
from IPython.core.debugger import set_trace

os.environ['TRITON_INTERPRET'] = '1'

def check_tensors_gpu_ready(*tensors):
  for t in tensors:
    assert t.is_contiguous, "A tensor is not contiguous"
    if not os.environ.get('TRITON_INTERPRET') == '1': assert t.is_cuda, "A tensor is not on cuda"

def test_pid_conds(conds, pid_0=[0], pid_1=[0], pid_2=[0]):
  '''Test if condition on pids are fulfilled
  E.g.:
    '=0' checks that pid_0 == 0
    '>1' checks that pid_1 > 1
    '>1,=0' checks that pid_0 > 1 and pid_1 == 0
  '''
  pids = pid_0[0], pid_1[0], pid_2[0]
  conds = conds.replace(' ', '').split(',')
  for i, (cond, pid) in enumerate(zip(conds, pids)):
    if cond=='': continue
    op, threshold = cond[0], int(cond[1:])
    if op not in ['<', '>', '>=', '<=', '=', '!=']: raise ValueError(f"Rules may only use these ops: '<', '>', '>=', '<=', '=', '!='. Invalid rule: '{condition}'.")
    op = '==' if op == '=' else op
    if not eval(f'{pid} {op} {threshold}'): return False
  return True

assert test_pid_conds('')
assert test_pid_conds('>0', [1], [1])
assert not test_pid_conds('>0', [0], [1])
assert test_pid_conds('=0,=1', [0], [1], [0])

def breakpoint_if(conds, pid_0=[0], pid_1=[0], pid_2=[0]):
  '''Stop kernel, if any condition of pids is fulfilled'''
  if test_pid_conds(conds, pid_0, pid_1, pid_2): set_trace()

def print_if(txt, conds, pid_0=[0], pid_1=[0], pid_2=[0]):
  '''Print txt, if any condition of pids is fulfilled'''
  if test_pid_conds(conds, pid_0, pid_1, pid_2): print(txt)

def cdiv(a, b): return (a + b - 1) // b
assert cdiv(10, 2) == 5
assert cdiv(10, 3) == 4

In [6]:
import torch
import triton
import triton.language as tl

# Copying a tensor

In [8]:
def copy(x, bs, kernel_fn):
  z = torch.zeros_like(x)
  check_tensors_gpu_ready(x, z)
  n = x.numel()
  n_blocks = cdiv(n, bs)
  grid = (n_blocks, )

  kernel_fn[grid](x, z, n, bs)

  return z

In [9]:
@triton.jit
def copy_k(x_ptr, z_ptr, n, bs: tl.constexpr):
  pid = tl.program_id(0)
  offs = pid * bs + tl.arange(0, bs)
  mask = offs < n
  x = tl.load(x_ptr + offs, mask)
  tl.store(z_ptr + offs, x, mask)
  print_if(f'pid = {pid} | offs = {offs}, mask = {mask}, x = {x}', '')


In [12]:
z = copy(x, bs=2, kernel_fn=copy_k)

pid = [0] | offs = [0 1], mask = [ True  True], x = [1 2]
pid = [1] | offs = [2 3], mask = [ True  True], x = [3 4]
pid = [2] | offs = [4 5], mask = [ True  True], x = [5 6]


In [13]:
x, z

(tensor([1, 2, 3, 4, 5, 6]), tensor([1, 2, 3, 4, 5, 6]))